In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

In [ ]:
experiment_name = "orca"
base_path = Path("../..")
data_dir = base_path / "data"
mixing_dir = data_dir / "data_mixing" / experiment_name

orca_path = data_dir / "orca.json"
with orca_path.open("r", encoding="utf-8") as read_file:
    data_s = json.load(read_file)

## Sample validation dataset

In [ ]:
import json
import random
from collections import defaultdict

data_path = data_dir / "orca_copy.json"

with open(data_path, "r") as f:
    data = json.load(f)

# (Optional) Ensure each item has a truly unique ID:
import uuid
for item in data:
    if "unique_id" not in item:
        item["unique_id"] = str(uuid.uuid4())

# Build the sources dictionary by domain:
sources = defaultdict(list)
for item in data:
    domain = item["id"]  # or item["domain"] if your data uses that key
    sources[domain].append(item)

val_output_tokens_per_domain = 150000  # Rough target per domain
validation_data = defaultdict(list)
validation_items = []
source_list = ["t0", "cot", "flan", "niv"]

for domain in source_list:
    random.shuffle(sources[domain])

    domain_val_items = []
    current_tokens = 0

    # 1) First pass selection
    for item in sources[domain]:
        if current_tokens >= val_output_tokens_per_domain:
            break
        domain_val_items.append(item)
        current_tokens += item["output_len"]

    # 2) If still under target, collect from remaining
    if current_tokens < val_output_tokens_per_domain:
        remaining_items = [
            itm for itm in sources[domain]
            if itm not in domain_val_items
        ]
        random.shuffle(remaining_items)

        for item in remaining_items:
            if current_tokens >= val_output_tokens_per_domain:
                break
            domain_val_items.append(item)
            current_tokens += item["output_len"]

    # Store validation data for this domain
    validation_data[domain] = domain_val_items
    validation_items.extend(domain_val_items)

    # >>> Use a set of unique IDs to remove from sources <<<
    val_ids = {itm["unique_id"] for itm in domain_val_items}
    sources[domain] = [itm for itm in sources[domain] if itm["unique_id"] not in val_ids]

    print(
        f"Domain '{domain}': Selected {len(domain_val_items)} items "
        f"with total output tokens {current_tokens}"
    )

print("\nValidation splits created for each domain.")
print(f"Total validation items across all domains: {len(validation_items)}")

for domain in source_list:
    for item in validation_data[domain]:
        del item["input_len"]
        del item["output_len"]
        del item["unique_id"]
        del item["id"]

for domain in source_list:
    filename = mixing_dir / f"orca_{domain}_val.json"
    with filename.open("w", encoding="utf-8") as f:
        json.dump(validation_data[domain], f, indent=2)
    print(f"Saved validation data for domain '{domain}' to '{filename}'.")

# Save leftovers to train file
training_list = []
for domain in source_list:
    training_list.extend(sources[domain])

with (mixing_dir / "orca_sample.json").open("w", encoding="utf-8") as f:
    json.dump(training_list, f, indent=2)

print("Remaining (training) data saved.")

Domain 't0': Selected 1407 items with total output tokens 150215
Domain 'cot': Selected 1022 items with total output tokens 150086
Domain 'flan': Selected 1164 items with total output tokens 150171
Domain 'niv': Selected 1010 items with total output tokens 150058

Validation splits created for each domain.
Total validation items across all domains: 4603
Saved validation data for domain 't0' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_t0_val.json'.
Saved validation data for domain 'cot' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_cot_val.json'.
Saved validation data for domain 'flan' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_flan_val.json'.
Saved validation data for domain 'niv' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_niv_val.json'.
Remaining (training) data saved.


In [ ]:
import json
import os
import random
from pathlib import Path

def load_data(file_path):
    """
    Load JSON data from a file.

    Parameters:
    - file_path (str | Path): Path to the JSON file.

    Returns:
    - data (list of dict): The loaded JSON data.

    Raises:
    - FileNotFoundError: If the file does not exist.
    - json.JSONDecodeError: If the file contains invalid JSON.
    """
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    with file_path.open("r", encoding="utf-8") as read_file:
        data = json.load(read_file)

    return data

def sample_tokens(data, limit, seed=42, max_passes=10):
    """
    Shuffle the data multiple times (upsampling) and sample items until the cumulative tokens
    (input_len + output_len) meets or exceeds the 'limit'. If all data is used and the limit
    is not yet met, reshuffle and continue sampling until 'max_passes' is reached.

    Once an item causes the cumulative tokens to exceed the limit, include that item and stop.
    After sampling, remove the 'output_len', 'input_len', and 'domain' keys from the items.

    Parameters:
    - data (list of dict): The input data to sample from.
    - limit (int): The token limit.
    - seed (int): The base random seed for shuffling.
    - max_passes (int): Maximum number of times to pass through the data for upsampling.

    Returns:
    - list of dict: The sampled items with specified keys removed.
    """
    total_tokens = 0
    sampled_items = []
    pass_num = 0  # To track the number of times we've looped through the data

    while total_tokens < limit and pass_num < max_passes:
        # Update the seed for each pass to ensure different shuffles
        current_seed = seed + pass_num
        random.seed(current_seed)

        shuffled_data = data.copy()
        random.shuffle(shuffled_data)

        for item in shuffled_data:
            item_tokens = item["output_len"] + item["input_len"]
            if total_tokens + item_tokens <= limit:
                sampled_items.append(item.copy())
                total_tokens += item_tokens
            else:
                # Adding this item would exceed the limit, include it and stop
                sampled_items.append(item.copy())
                total_tokens += item_tokens
                # After adding the item, exit all loops
                break
        else:
            # Completed a full pass without exceeding the limit
            pass_num += 1
            continue  # Continue to the next pass if the limit is not yet met
        break  # Exit the while loop after the break in for-loop

    if total_tokens < limit:
        print(f"Warning: Reached maximum passes ({max_passes}) without meeting token limit.")

    # Remove specified keys from sampled items
    for item in sampled_items:
        del item["output_len"]
        del item["input_len"]
        del item["id"]
        del item["unique_id"]

    return sampled_items

def ensure_directory_exists(directory_path):
    """
    Ensure that the specified directory exists. If not, create it.

    Parameters:
    - directory_path (str | Path): Path to the directory.
    """
    Path(directory_path).mkdir(parents=True, exist_ok=True)

def main():
    # Define file paths and parameters
    data_file_path = mixing_dir / f"{experiment_name}_sample.json"
    output_dir = mixing_dir

    # Load data
    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return

    # Define token limits
    token_limits = {
        "t0": 168090000,
        "cot": 7410000,
        "flan": 106830000,
        "niv": 17670000,
    }

    # Define domains to process
    domains = ["t0", "cot", "flan", "niv"]
    total_data = []
    # Ensure the output directory exists
    ensure_directory_exists(output_dir)

    for domain in domains:
        # Filter the data by domain
        domain_data = [item for item in data if item["id"] == domain]

        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue

        limit = token_limits[domain]
        # Sample tokens with upsampling
        sampled_items = sample_tokens(domain_data, limit)
        total_data.extend(sampled_items)

        # Print stats
        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")

    with (output_dir / f"{experiment_name}_original.json").open("w", encoding="utf-8") as wri_f:
        json.dump(total_data, wri_f, indent=4)

if __name__ == "__main__":
    main()

Domain: t0 | Limit: 168,090,000 | Items: 381417
Domain: cot | Limit: 7,410,000 | Items: 32053
Domain: flan | Limit: 106,830,000 | Items: 324354
Domain: niv | Limit: 17,670,000 | Items: 47421


## Sample equal for orca

In [ ]:
def run_orca_equal():
    # Define file paths and parameters
    data_file_path = mixing_dir / f"{experiment_name}_sample.json"
    output_dir = mixing_dir

    # Load data
    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return

    # Define token limits
    token_limits = {
        "t0": 75000000,
        "cot": 75000000,
        "flan": 75000000,
        "niv": 75000000,
    }

    # Define domains to process
    domains = ["t0", "cot", "flan", "niv"]
    total_data = []
    # Ensure the output directory exists
    ensure_directory_exists(output_dir)

    for domain in domains:
        # Filter the data by domain
        domain_data = [item for item in data if item["id"] == domain]

        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue

        limit = token_limits[domain]
        # Sample tokens with upsampling
        sampled_items = sample_tokens(domain_data, limit)
        total_data.extend(sampled_items)

        # Print stats
        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")

    with (output_dir / f"{experiment_name}_equal.json").open("w", encoding="utf-8") as wri_f:
        json.dump(total_data, wri_f, indent=4)

run_orca_equal()

Domain: t0 | Limit: 75,000,000 | Items: 170899
Domain: cot | Limit: 75,000,000 | Items: 325228
Domain: flan | Limit: 75,000,000 | Items: 228179
Domain: niv | Limit: 75,000,000 | Items: 201489


In [ ]:
dataset_info_path = data_dir / "dataset_info.json"
with dataset_info_path.open("r", encoding="utf-8") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

dataset_name = "orca_equal"
output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
dataset_info[dataset_name] = {
    "file_name": output_path
}

with dataset_info_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

## Sample 660K

In [ ]:
def generate_orca_scaled_samples(base_token=660_000):
    data_file_path = mixing_dir / "orca_sample.json"
    output_dir = mixing_dir

    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return

    token_limits = {
        "one": base_token,
        "half": base_token // 2,
        "third": base_token // 3,
        "double": base_token * 2,
        "triple": base_token * 3,
    }

    domains = ["t0", "cot", "flan", "niv"]
    ensure_directory_exists(output_dir)

    for domain in domains:
        domain_data = [item for item in data if item["id"] == domain]

        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue

        for name, limit in token_limits.items():
            sampled_items = sample_tokens(domain_data, limit)
            out_path = output_dir / f"{base_token}_{domain}_{name}.json"

            try:
                with out_path.open("w", encoding="utf-8") as f:
                    json.dump(sampled_items, f, indent=4)
            except OSError as e:
                print(f"Error writing to file {out_path}: {e}")
                continue

            print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")


generate_orca_scaled_samples()

Domain: t0 | Limit: 660,000 | Items: 1540
Domain: t0 | Limit: 330,000 | Items: 751
Domain: t0 | Limit: 220,000 | Items: 507
Domain: t0 | Limit: 1,320,000 | Items: 3039
Domain: t0 | Limit: 1,980,000 | Items: 4563
Domain: cot | Limit: 660,000 | Items: 2870
Domain: cot | Limit: 330,000 | Items: 1447
Domain: cot | Limit: 220,000 | Items: 960
Domain: cot | Limit: 1,320,000 | Items: 5740
Domain: cot | Limit: 1,980,000 | Items: 8600
Domain: flan | Limit: 660,000 | Items: 1975
Domain: flan | Limit: 330,000 | Items: 1020
Domain: flan | Limit: 220,000 | Items: 683
Domain: flan | Limit: 1,320,000 | Items: 4003
Domain: flan | Limit: 1,980,000 | Items: 6048
Domain: niv | Limit: 660,000 | Items: 1778
Domain: niv | Limit: 330,000 | Items: 907
Domain: niv | Limit: 220,000 | Items: 584
Domain: niv | Limit: 1,320,000 | Items: 3535
Domain: niv | Limit: 1,980,000 | Items: 5309


In [ ]:
dataset_info_path = data_dir / "dataset_info.json"
with dataset_info_path.open("r", encoding="utf-8") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

domains = ["t0", "cot", "flan", "niv"]
base_token = 660_000
token_limits = {
    "one": base_token,
    "half": base_token // 2,
    "third": base_token // 3,
    "double": base_token * 2,
    "triple": base_token * 3,
}
experiment_name = "orca"
for domain in domains:
    for size in token_limits.keys():
        dataset_name = f"{base_token}_{domain}_{size}"
        output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
        dataset_info[dataset_name] = {
            "file_name": output_path
        }

with dataset_info_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

## Our method

In [ ]:
def run_orca_qwen_allocation():
    data_file_path = mixing_dir / f"{experiment_name}_sample.json"
    output_dir = mixing_dir

    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as err:
        print(f"Error loading data: {err}")
        return

    rate = [0.35983086, 0.20161636, 0.34249649, 0.096]
    token_limits = {
        "t0": int(300_000_000 * rate[0]),
        "cot": int(300_000_000 * rate[1]),
        "flan": int(300_000_000 * rate[2]),
        "niv": int(300_000_000 * rate[3]),
    }

    domains = ["t0", "cot", "flan", "niv"]
    ensure_directory_exists(output_dir)

    total_data = []
    for domain in domains:
        domain_data = [item for item in data if item["id"] == domain]
        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue

        limit = token_limits[domain]
        sampled_items = sample_tokens(domain_data, limit)
        total_data.extend(sampled_items)
        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")

    output_path = output_dir / f"{experiment_name}_Qwen_ours.json"
    with output_path.open("w", encoding="utf-8") as wri_f:
        json.dump(total_data, wri_f, indent=4)

run_orca_qwen_allocation()

Domain: t0 | Limit: 107,949,257 | Items: 245344
Domain: cot | Limit: 60,484,908 | Items: 262431
Domain: flan | Limit: 102,748,947 | Items: 311968
Domain: niv | Limit: 28,800,000 | Items: 77227


In [ ]:
dataset_info_path = data_dir / "dataset_info.json"
with dataset_info_path.open("r", encoding="utf-8") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

experiment_name = "orca"

dataset_name = "orca_Qwen_ours"
relative_output = Path("data_mixing") / experiment_name / f"{dataset_name}.json"
dataset_info[dataset_name] = {
    "file_name": str(relative_output)
}

with dataset_info_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)